### Замер времени выполения

In [1]:
%load_ext autotime
%matplotlib inline

time: 473 ms (started: 2026-05-24 16:02:38 +03:00)


In [2]:
import time

start_time = time.perf_counter()

time: 608 μs (started: 2026-05-24 16:02:39 +03:00)


### Подключаем основные модули

In [3]:
import socket

import clickhouse_connect
import matplotlib.pyplot as plt
import my_modules.my_describe as my_describe
import my_modules.my_modules as mm
import pandas as pd
import seaborn as sns
from cl_wrapper import connection, sql
from IPython.display import display


time: 197 ms (started: 2026-05-24 16:02:39 +03:00)


### Подключаемся к база данных `clickhouse`

Подключение к ClickHouse через официальный драйвер `clickhouse-connect`<br>
Если запуск `.ipynb` происходит на локальной машине то `host = "localhost"`.<br>
Иначе указывается адрес сурвиса в рамках docker - `"service.db_clickhouse"` (укажите свой). 

Используется небольшая самописная обертка для `sql запросов` - `cl_wrapper`

In [4]:
hostname = socket.gethostname()
host = "service.db_clickhouse"
if hostname == "home-NMH-WDX9":
    host = "localhost"

client = connection.Connection(host=host)
conn = client.connect()

query = sql.SQL(conn=conn)
# set db and table globally or use set_db and set_table on query
# query.set_db("cars").set_table("car_sales").get_total()  # noqa: ERA001
query.db = "cars"
query.table = "car_sales"


time: 18.9 ms (started: 2026-05-24 16:02:39 +03:00)


### Получаем общее кол-во записей в таблице `car_sales`

Получаем общее кол-во записей в таблице `car_sales`. 

На основании полученной информаци формируем размер выборки `sample_size`. <br>
Размер выборки, в данном случае, формируется в размере `0.1%` процента от общего кол-ва записей в таблице.

In [5]:
rows_count = (
    # query.set_db("cars").set_table("car_sales").get_total().query().result_rows[0][0]  # noqa: ERA001
    query.get_total().query().result_rows[0][0]
)
sample_size = round(rows_count * 0.01)
print(f"Записей = {rows_count:_d}, Размер выборки = {sample_size:_d}")

Записей = 1_294_757, Размер выборки = 12_948
time: 6.41 ms (started: 2026-05-24 16:02:39 +03:00)


### Формирование отчета аналогичного `pandas info`

In [6]:
result = query.get_describe_table().query_df().set_index("name")

cars_columns = result.index.to_list()


field_not_null_counts = {}
for column in cars_columns:
    rez = query.get_count_column_is_not_null(column).query()
    field_not_null_counts[column] = rez.result_rows[0][0]

df = pd.DataFrame.from_dict(
    field_not_null_counts, orient="index", columns=["Non-Null Count"]
)
df["Null Count"] = rows_count - df["Non-Null Count"]
df["Null Count %"] = round((df["Null Count"] * 100) / rows_count, 2)

rez = pd.concat([result, df], axis=1).reset_index().rename(columns={"index": "column"})
rez_styled = rez.style.map(mm.highlight_values, subset=["Null Count %"]).format({
    "Null Count %": "{:.2f}"
})

rez_styled

,column,type,default_type,default_expression,comment,codec_expression,ttl_expression,Non-Null Count,Null Count,Null Count %
0,brand,Nullable(String),,,,,,1294757,0,0.00
1,name,Nullable(String),,,,,,1294757,0,0.00
2,bodyType,Nullable(String),,,,,,1294757,0,0.00
3,color,Nullable(String),,,,,,1257029,37728,2.91
4,fuelType,Nullable(String),,,,,,1289815,4942,0.38
5,year,Nullable(UInt32),,,,,,724644,570113,44.03
6,mileage,Nullable(UInt32),,,,,,771799,522958,40.39
7,transmission,Nullable(String),,,,,,1289563,5194,0.40
8,power,Nullable(UInt16),,,,,,1273353,21404,1.65
9,price,UInt32,,,,,,1294757,0,0.00


time: 230 ms (started: 2026-05-24 16:02:39 +03:00)


In [20]:
cars_columns2 = cars_columns.copy()
cars_columns2.remove("link")
cars_columns2.remove("description")

dfs = []
sample_count = 3
df = query.get_samples(
    cars_columns=cars_columns2,
    sample_count=3,
    sample_size=sample_size,
).query_df()
df_renamed = df.rename(columns={"engineDisplacement": "ED"})
for group_id in range(1, sample_count + 1):
    dfs.append(df_renamed[df_renamed["tile"] == group_id])  # noqa: PERF401


time: 1.83 s (started: 2026-05-24 16:17:03 +03:00)


In [8]:
dfs[0].sample(5)

,brand,name,bodyType,color,fuelType,year,mileage,transmission,power,price,vehicleConfiguration,engineName,ED,date,location,parse_date,tile,row_num
5967,Лада,2114 Самара,Хэтчбек 5 дв.,Зеленый,Бензин,2003,195000,Механика,77,135000,1.5 MT Базовая,ВАЗ-2111,1.5,2023-04-30,Пермь,2023-05-06 01:00:00,1,6606
4881,Лада,1111 Ока,Хэтчбек 3 дв.,Красный,Бензин,1996,<NA>,Механика,33,70000,0.6 MT 11113,ВАЗ-11113,0.7,2023-06-09,Ялуторовск,2023-06-09 22:00:00,1,2530
8611,Лада,2107,Седан,Синий,Бензин,2001,11000,Механика,74,175000,1.6 MT 21074-30-011,ВАЗ-21067-10,1.6,2023-05-11,Улан-Удэ,2023-05-14 20:00:00,1,8980
7511,Toyota,Wish,Минивэн,Черный,Бензин,2010,<NA>,Вариатор,144,1060000,1.8 S,2ZR-FAE,1.8,2023-06-08,Якутск,2023-06-08 04:00:00,1,8708
3185,Chery,Tiggo 8 Pro,Джип 5 дв.,Красный,Бензин,2021,<NA>,Робот,186,3659900,1.6 DCT Ultimate,SQRF4J16,1.6,2023-06-15,Москва,2023-06-15 18:00:00,1,3781


time: 15.6 ms (started: 2026-05-24 16:02:41 +03:00)


In [9]:
%%time
fields = ["price", "power", "year"]
agg_funcs = {
    "count": "count",
    "mean": "avg",
    "std": "stddevPop",
    "min": "min",
    "25%": "quantile(0.25)",
    "50%": "quantile(0.50)",
    "75%": "quantile(0.75)",
    "max": "max",
}
totals = {}
for field in fields:
    totals[f"total_{field}"] = {}
    for af in agg_funcs:
        sql = f"""SELECT {agg_funcs[af]}({field}) FROM {db}.{table};"""  # noqa: S608
        result = client.query(sql)
        totals[f"total_{field}"].setdefault(af, 0)
        totals[f"total_{field}"][af] = result.result_rows[0][0]
totals = pd.DataFrame(totals)
with pd.option_context(
    "display.float_format",
    "{:.2f}".format,
    "display.expand_frame_repr",
    False,
):
    display(totals)

CPU times: user 14 μs, sys: 1 μs, total: 15 μs
Wall time: 17.6 μs


NameError: name 'db' is not defined

time: 94.6 ms (started: 2026-05-24 16:02:41 +03:00)


### Получение описательной статистики

In [ ]:
%%time
fields = ["price", "power", "year"]
descs = []
for i in range(sample_count):
    desc = dfs[i][fields].describe()
    desc.columns = [f"df{i + 1}_{col}" for col in desc.columns]
    descs.append(desc)

combined_desc = pd.concat(descs, axis=1)

combined_desc1 = my_describe.custom_describe(
    totals, combined_desc, sample_count, fields
)

with pd.option_context(
    "display.float_format",
    "{:.2f}".format,
    "display.expand_frame_repr",
    False,
):
    display(combined_desc1)

In [ ]:
%%time
dfs[0].info()

In [ ]:
%%time
total_rows, total_null_val, types, table = mm.custom_info(dfs[0])
with pd.option_context(
    "display.expand_frame_repr",
    False,
):
    display(table)
    print(
        f"Всего записей: {total_rows}, Null значений: {total_null_val}, Типы данных: {types}"
    )

In [ ]:
# %%time

# cars_colums = df.columns.to_list()
# # cars_colums.remove("link")
# # cars_colums.remove("description")

# dfs = []
# sample_size_per_group = 10000

# for group_id in range(1, 4):
#     query = f"""
#     WITH ranked AS (
#         SELECT {','.join(cars_colums)}, NTILE(3) OVER (ORDER BY rand()) AS tile
#         FROM cars.car_sales
#     )
#     SELECT * FROM ranked
#     WHERE tile = {group_id}
#     ORDER BY rand()
#     LIMIT {sample_size_per_group}
#     """

#     dfs.append(client.query_df(query))


In [ ]:
%%time
plt.figure(figsize=(16, 6))
for idx, df_number in enumerate(dfs, 1):
    sns.lineplot(
        x="date",
        y="price",
        data=df_number,
        linewidth=2.5,
        errorbar="sd",
        label=f"Выборка #{idx}",
    )
plt.xlabel("Дата")
plt.ylabel("Цена")
plt.show()

In [ ]:
%%time

for idx, df_number in enumerate(dfs):
    fig, axes = plt.subplots(1, 4, figsize=(15, 3), constrained_layout=True)
    fig.suptitle(f"Графики для выборки #{idx + 1}", fontsize=16, fontweight="bold")

    # График 1: power vs price
    sns.regplot(data=dfs[idx], x="power", y="price", ax=axes[0])
    axes[0].set_xlabel("Мощность (л.с.)")
    axes[0].set_ylabel("Цена")
    axes[0].set_title("Зависимость цены от мощности")

    # График 2: mileage vs price
    sns.regplot(data=dfs[idx], x="mileage", y="price", ax=axes[1])
    axes[1].set_xlabel("Пробег (км)")
    axes[1].set_ylabel("Цена")
    axes[1].set_title("Зависимость цены от пробега")

    # График 3: year vs price
    sns.regplot(data=dfs[idx], x="year", y="price", ax=axes[2])
    axes[2].set_xlabel("Год выпуска")
    axes[2].set_ylabel("Цена")
    axes[2].set_title("Зависимость цены от года выпуска")

    # График 4: year vs power
    sns.regplot(data=dfs[idx], x="year", y="power", ax=axes[3])
    axes[3].set_xlabel("Год выпуска")
    axes[3].set_ylabel("Мощность (л.с.)")
    axes[3].set_title("Зависимость мощности от года выпуска")

plt.show()


In [ ]:
# # dsf["car_sales"]["year"].isnull().sum()
# df_test = cars[cars["year"].isnull()]
# df_test

In [ ]:
# df1 = dfs[0][['price', 'year', 'mileage', 'power']]
# g = sns.PairGrid(df1, diag_sharey=False)
# g.map_upper(sns.regplot)
# g.map_lower(sns.regplot)
# g.map_diag(sns.kdeplot, lw=2)
# plt.show()

In [ ]:
end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Общее время выполнения: {total_time:.4f} секунд")